In [20]:
#Read the spell file
import pandas
import re
# Read the spells from the Spells.csv document. The file is formatted as Spell ID,Incantation,Spell Name,Effect,Light
# Create a list of all incantations.
incantation_list = pandas.read_csv("spells.csv", sep=",")["Incantation"].dropna().str.lower().tolist()
# there are 61 spells total. Display list of spells.
#print(f"There are {len(incantation_list)} spells in the list.")
# Show the first 10 spells in the list.
#print("The first 10 spells in the list are:")
#print(incantation_list[:10])


# There are 125 relevant characters in the character sheet. For the creation of our good and evil spectrum, we will train with 2/3rds of the set.
# For training, we will use characters 0,1,2 and 8 (Harry, Ron, Hermione and Voldemort, 0 indexing), alongside 77 other characters picked at random. The remaining 41 characters will be used for testing.
# the file is formatted as Character ID,Character Name,Species,Gender,House,Patronus,Wand (Wood),Wand (Core)
try:
    character_df = pandas.read_csv("characters.csv", sep=",", encoding="utf-8")
except UnicodeDecodeError:
    character_df = pandas.read_csv("characters.csv", sep=",", encoding="cp1252")

character_df = character_df.dropna(subset=["Character Name"])
#print(f"There are {len(character_df)} characters in the list.")
#print("The first 10 characters in the list are:")
#print(character_df["Character Name"][:10])


# Finally, we have the entire script for the movies in Dialogue.csv, formatted as Dialogue ID,Chapter ID,Place ID,Character ID,Dialogue
# For every line of dialogue, check of it contains any incantation from our list. Keep track of counts of how often it appears globally, and on a person basis, 
# I only care about the character ID and the dialogue section.
try:
    dialogue = pandas.read_csv("dialogue.csv", sep=",", encoding="utf-8", quoting=3, on_bad_lines='skip')
except UnicodeDecodeError:
    dialogue = pandas.read_csv("dialogue.csv", sep=",", encoding="cp1252", quoting=3, on_bad_lines='skip')

dialogue = dialogue[["Character ID", "Dialogue"]]
# Read the first 5 lines.
#print("The first 5 lines of dialogue are:")
#print(dialogue.head())



The above code handles initialization of the relevant datasets, containing lists if spells (specifically, incantations), All relevant cast members, and the script for the movies. With this parsed we can count how often an incantation occurs, and who casts it. This will be relevant for our analysis going forward.

In [22]:
# The dialogue is now parsable, with the relevant speaker character ID.
# Now, we sieve through the dialogue, looking for incantations, keeping track of global use, and per-person use.
incantation_counts = {incantation: 0 for incantation in incantation_list}
character_incantation_counts = {character_id: {incantation: 0 for incantation in incantation_list} for character_id in character_df["Character ID"]}

# Iterate through dialogue and count incantations
for idx, row in dialogue.iterrows():
    dialogue_text = row["Dialogue"].lower()
    character_id = row["Character ID"]
    
    for incantation in incantation_list:
        if incantation in dialogue_text:
            incantation_counts[incantation] += 1
            character_incantation_counts[character_id][incantation] += 1

# Create a summary of spell usage by character
spell_usage = incantation_counts
character_spell_usage = character_incantation_counts

# Sort spells by usage
sorted_spells = sorted(spell_usage.items(), key=lambda x: x[1], reverse=True)
print("Most used spells:")
for spell, count in sorted_spells[:4]:  # Display top 4 spells
    print(f"{spell}: {count} times")

# Sort characters by total spell usage
character_totals = {char: sum(spells.values()) for char, spells in character_spell_usage.items()}
sorted_characters = sorted(character_totals.items(), key=lambda x: x[1], reverse=True)
print("\nMost spell-casting characters:")
for character_id, total in sorted_characters[:4]:  # Display top 4 characters
    character_name = character_df.loc[character_df["Character ID"] == character_id, "Character Name"].values
    if len(character_name) > 0:
        print(f"{character_name[0]}: {total} spells cast")
        
        
# Show the list of spells that Voldemort casts.
voldemort_rows = character_df[
    character_df["Character Name"].str.contains("voldemort", case=False, na=False)
]

if voldemort_rows.empty:
    print("\nVoldemort not found in character_df.")
else:
    print("\nSpells cast by Voldemort:")
    for voldemort_id in voldemort_rows["Character ID"].tolist():
        voldemort_name = character_df.loc[
            character_df["Character ID"] == voldemort_id, "Character Name"
        ].iloc[0]
        voldemort_spells = character_spell_usage.get(voldemort_id, {})
        non_zero_spells = [(spell, count) for spell, count in voldemort_spells.items() if count > 0]

        print(f"\n{voldemort_name} (ID {voldemort_id}):")
        if non_zero_spells:
            for spell, count in sorted(non_zero_spells, key=lambda x: x[1], reverse=True):
                print(f"{spell}: {count} times")
        else:
            print("No detected spell incantations in dialogue.")
            
# Show the list of spells that Harry Potter casts.
harry_rows = character_df[
    character_df["Character Name"].str.contains("harry potter", case=False, na=False)
]

if harry_rows.empty:
    print("\nHarry Potter not found in character_df.")
else:
    print("\nTop 5 spells cast by Harry Potter:")
    for harry_id in harry_rows["Character ID"].tolist():
        harry_name = character_df.loc[
            character_df["Character ID"] == harry_id, "Character Name"
        ].iloc[0]
        harry_spells = character_spell_usage.get(harry_id, {})
        non_zero_spells = [(spell, count) for spell, count in harry_spells.items() if count > 0]

        print(f"\n{harry_name} (ID {harry_id}):")
        if non_zero_spells:
            for spell, count in sorted(non_zero_spells, key=lambda x: x[1], reverse=True)[:5]:
                print(f"{spell}: {count} times")
        else:
            print("No detected spell incantations in dialogue.")

Most used spells:
expecto patronum: 11 times
expelliarmus: 11 times
lumos: 11 times
accio: 7 times

Most spell-casting characters:
Harry Potter: 51 spells cast
Hermione Granger: 29 spells cast
Ron Weasley: 6 spells cast
Albus Dumbledore: 5 spells cast

Spells cast by Voldemort:

Voldemort (ID 9):
avada kedavra: 2 times

Top 5 spells cast by Harry Potter:

Harry Potter (ID 1):
expecto patronum: 9 times
lumos: 7 times
accio: 4 times
expelliarmus: 3 times
stupefy: 3 times


In the output, we can see that Expecto Patronum is the most commonly used spell, also, harry is respknsible for 9 of the 11 times it is used. Harry is responsible for most spells cast in the series, for obvious reasons.
With the spells now correctly accounted for, we must create a scoring system  for the spells, to place them on an 'evil-ness' scale. Charms, Hexes, Curses and so on are classes we can use for general classification, and then, by hand, especially important incantations can get more in-depth scores.

Scoring system:
Curses are evil. Within the curses, are especially evil curses, marked as unforgivable Avada Kedavra(instant death), Crucio (torture). A long list of spells are relatively neutral, Lumos (create light). Healing spells also exist, and are counted as 'good' spells. Expelliarmus (disarming) may also count as a 'good' spell, as it tries to disengage a fight without casualty. 